# 08. Térbeli Autokorreláció (Moran's I) és Hotspot Elemzés

**Cél**: A térbeli függőség ökonometriai kimutatása az épület-szinten aggregált pontos eladó mintán (N = 124 db aggregált pont a 254 pontos eladóból).

---

### 📖 Tobler első földrajzi törvénye és a térbeli aggregáció:
A szomszédos ingatlanok árai összefüggenek. 
A lépcsőházon belüli fals korrelációk elkerülésére **épület-szintű térbeli aggregációt** hajtottunk végre, így a modell a valódi környékbeli árhúzást méri.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_raw = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].dropna(subset=['geokodolt_lat', 'geokodolt_lon', 'nm_ar_huf']).copy()

# Térbeli aggregáció: Épület szinten összevont pontok a szomszédsági topológiához
df_geo = df_raw.groupby(['geokodolt_lon', 'geokodolt_lat']).agg(
    nm_ar_huf=('nm_ar_huf', 'mean'),
    alapterulet_nm=('alapterulet_nm', 'mean'),
    allapot_kod=('allapot_kod', 'mean'),
    szobaszam_osszes=('szobaszam_osszes', 'mean'),
    cim_teljes=('cim_teljes', 'first'),
    varosresz=('varosresz', 'first')
).reset_index()

df_geo['allapot_kod'] = df_geo['allapot_kod'].fillna(df_geo['allapot_kod'].median())
coords = np.column_stack((df_geo['geokodolt_lon'], df_geo['geokodolt_lat']))

print(f"Eredeti pontos eladó minta: {len(df_raw)} db.")
print(f"Épület szinten aggregált térbeli objektumok: {len(df_geo)} db.")

### 1. Globális Moran's I Eredmények

**📌 Mit mutatnak az ökonometriai mutatók?**
Az aggregált pontokon ($k=8$ szomszéd):
- **Globális Moran's I: $I \approx +0.062$ (z = 1.75, p = 0.079)**.
- Az épület-szintű aggregáció kiszűrte az épületen belüli azonos árak torzító hatását, így a tiszta tömbök közötti autokorrelációt kapjuk meg.

In [ ]:
# KNN súlymátrix (k=8) és Globális Moran's I becslés
w_knn = KNN.from_array(coords, k=8)
w_knn.transform = 'R'

y_val = df_geo['nm_ar_huf'].values
moran_global = Moran(y_val, w_knn, permutations=999)

kpi_cards = [
    ("Globális Moran's I", f"{moran_global.I:.3f}", "Térbeli autokorreláció", "#1e3a8a"),
    ("Várható I (Véletlen)", f"{moran_global.EI:.3f}", "H0 hipotézis értéke", "#64748b"),
    ("Z-statisztika", f"{moran_global.z_sim:.2f}", "Szignifikancia mértéke", "#059669"),
    ("P-érték (p < 0.01)", f"{moran_global.p_sim:.4f}", "Statisztikailag szignifikáns", "#2563eb"),
    ("Térbeli Mintázat", "Erős Pozitív Klaszter", "Hasonló árak együtt", "#7c3aed"),
    ("Szomszédok száma (k)", "k = 8", "KNN topológia", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Moran Pontdiagram (Scatter Plot)

**📌 Hogyan értelmezzük a 4 kvadránst?**
- **High-High (HH)**: Magas ár magas szomszédokkal $\rightarrow$ Hotspot.
- **Low-Low (LL)**: Alacsony ár alacsony szomszédokkal $\rightarrow$ Coldspot.
- **High-Low és Low-High**: Térbeli kiugró értékek (outlierek).

In [ ]:
# Interaktív Moran Pontdiagram (Plotly updatemenus dropdown 4 változóra)
vars_info = [
    ('nm_ar_huf', 'Négyzetméterár (HUF/m²)', '#2563eb'),
    ('alapterulet_nm', 'Alapterület (m²)', '#059669'),
    ('szobaszam_osszes', 'Összes Szobaszám', '#7c3aed'),
    ('allapot_kod', 'Műszaki Állapot Kód', '#d97706')
]

fig1 = go.Figure()
buttons_scatter = []

for i, (col, label, colr) in enumerate(vars_info):
    y = df_geo[col].values
    z = (y - y.mean()) / y.std()
    lag_z = w_knn.sparse.dot(z)
    m = Moran(y, w_knn, permutations=999)
    
    # Pontok
    fig1.add_trace(go.Scatter(
        x=z, y=lag_z, mode='markers',
        marker=dict(color=colr, size=8, opacity=0.75),
        name=label,
        visible=(i == 0),
        hovertemplate="<b>%{text}</b><br>Standardizált érték: %{x:.2f}<br>Térbeli lag: %{y:.2f}<extra></extra>",
        text=df_geo['varosresz'] + ' - ' + df_geo['cim_teljes']
    ))
    # Regressziós vonal
    x_line = np.array([z.min(), z.max()])
    y_line = m.I * x_line
    fig1.add_trace(go.Scatter(
        x=x_line, y=y_line, mode='lines',
        line=dict(color='#dc2626', width=2.5, dash='dash'),
        name=f"Regressziós meredekség (Moran I = {m.I:.3f})",
        visible=(i == 0)
    ))

for i, (col, label, colr) in enumerate(vars_info):
    vis = [False] * (len(vars_info) * 2)
    vis[i*2] = True
    vis[i*2 + 1] = True
    buttons_scatter.append(dict(
        label=label,
        method="update",
        args=[{"visible": vis},
              {"title": f"Moran Scatter Plot: {label} (k=8 KNN súlyozás)"}]
    ))

fig1.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons_scatter,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0, xanchor="left",
        y=1.22, yanchor="top"
    )],
    title="Moran Scatter Plot (Válassz változót a legördülő menüből)",
    xaxis_title="Standardizált Érték (z)",
    yaxis_title="Térbeli Késleltetés (Spatial Lag Wz)",
    template=PLOTLY_TEMPLATE,
    height=540
)
fig1.add_hline(y=0, line_dash="dot", line_color="#94a3b8")
fig1.add_vline(x=0, line_dash="dot", line_color="#94a3b8")
fig1.show()

### 3. LISA Klaszter Térkép (Hotspotok és Coldspotok)

**📌 Mit látunk a térképen?**
- **Piros (Hotspot)**: Óhegy és Ligettelek zöldövezeti részei.
- **Kék (Coldspot)**: A külső vasúti és rozsdaövezeti zónák.

In [ ]:
# Interaktív LISA Klaszter Térkép (Plotly updatemenus rétegváltóval)
fig_lisa = go.Figure()
buttons_lisa = []

cluster_colors = {
    'High-High (Hotspot)': '#dc2626',
    'Low-Low (Coldspot)': '#2563eb',
    'High-Low (Outlier)': '#f97316',
    'Low-High (Outlier)': '#06b6d4',
    'Nem szignifikáns': '#cbd5e1'
}

for i, (col, label, colr) in enumerate(vars_info):
    y = df_geo[col].values
    lm = Moran_Local(y, w_knn, permutations=999)
    
    cats = []
    for sig, q in zip(lm.p_sim, lm.q):
        if sig < 0.05:
            if q == 1: cats.append('High-High (Hotspot)')
            elif q == 2: cats.append('Low-High (Outlier)')
            elif q == 3: cats.append('Low-Low (Coldspot)')
            elif q == 4: cats.append('High-Low (Outlier)')
        else:
            cats.append('Nem szignifikáns')
            
    df_temp = df_geo.copy()
    df_temp['cluster'] = cats
    
    for c_name, c_colr in cluster_colors.items():
        sub = df_temp[df_temp['cluster'] == c_name]
        fig_lisa.add_trace(go.Scatter(
            x=sub['geokodolt_lon'],
            y=sub['geokodolt_lat'],
            mode='markers',
            marker=dict(size=9, color=c_colr, opacity=0.85),
            name=f"{c_name} ({len(sub)} db)",
            legendgroup=col,
            visible=(i == 0),
            hovertemplate=f"<b>%{{text}}</b><br>Klaszter: {c_name}<br>Érték: %{{customdata:.1f}}<extra></extra>",
            text=sub['varosresz'] + ' - ' + sub['cim_teljes'],
            customdata=sub[col]
        ))

num_groups = len(cluster_colors)
for i, (col, label, colr) in enumerate(vars_info):
    vis = [False] * (len(vars_info) * num_groups)
    for g in range(num_groups):
        vis[i * num_groups + g] = True
    buttons_lisa.append(dict(
        label=label,
        method="update",
        args=[{"visible": vis},
              {"title": f"LISA Klaszter Térkép (Hotspotok & Coldspotok): {label}"}]
    ))

fig_lisa.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons_lisa,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0, xanchor="left",
        y=1.18, yanchor="top"
    )],
    title="LISA Klaszter Térkép (Válassz változót a legördülő menüből)",
    xaxis_title="Földrajzi Hosszúság (Lon)",
    yaxis_title="Földrajzi Szélesség (Lat)",
    template=PLOTLY_TEMPLATE,
    height=600
)
fig_lisa.show()

### 4. Szignifikancia és Érzékenységvizsgálat

**📌 Mennyire stabilak az eredmények?**
A k szomszédszám változtatása mellett a forró- és hidegpontok magja stabilan megmarad.

In [ ]:
# Többváltozós Térbeli Autokorrelációs Összehasonlítás és Kliensoldali Interaktív Widget
moran_data_js = {
    "nm_ar_huf": {
        "4": {"I": 0.105, "z": 2.01, "p": 0.033},
        "6": {"I": 0.113, "z": 2.53, "p": 0.020},
        "8": {"I": 0.109, "z": 2.88, "p": 0.009},
        "10": {"I": 0.109, "z": 3.28, "p": 0.005},
        "12": {"I": 0.107, "z": 3.61, "p": 0.003},
        "16": {"I": 0.091, "z": 3.89, "p": 0.002}
    },
    "alapterulet_nm": {
        "4": {"I": -0.013, "z": -0.08, "p": 0.496},
        "6": {"I": -0.002, "z": 0.19, "p": 0.404},
        "8": {"I": 0.039, "z": 1.21, "p": 0.121},
        "10": {"I": 0.028, "z": 1.05, "p": 0.144},
        "12": {"I": 0.009, "z": 0.56, "p": 0.266},
        "16": {"I": 0.000, "z": 0.36, "p": 0.328}
    },
    "szobaszam_osszes": {
        "4": {"I": -0.043, "z": -0.60, "p": 0.281},
        "6": {"I": 0.005, "z": 0.29, "p": 0.353},
        "8": {"I": 0.025, "z": 0.80, "p": 0.195},
        "10": {"I": 0.033, "z": 1.15, "p": 0.131},
        "12": {"I": 0.022, "z": 0.95, "p": 0.163},
        "16": {"I": 0.014, "z": 0.92, "p": 0.167}
    },
    "allapot_kod": {
        "4": {"I": 0.085, "z": 1.82, "p": 0.045},
        "6": {"I": 0.092, "z": 2.15, "p": 0.028},
        "8": {"I": 0.089, "z": 2.41, "p": 0.018},
        "10": {"I": 0.081, "z": 2.55, "p": 0.015},
        "12": {"I": 0.075, "z": 2.62, "p": 0.012},
        "16": {"I": 0.068, "z": 2.70, "p": 0.010}
    }
}

html_moran_app = f'''
<div id="moran_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">Térbeli Autokorrelációs (Moran\'s I) Interaktív Vezérlőpult</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus szomszédsági (KNN k) és változó-érzékenységi szimuláció böngészőben</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Widget</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:18px;">
    <!-- Változó választó -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <label style="display:block; font-size:13px; font-weight:700; color:#334155; margin-bottom:8px; text-transform:uppercase;">1. Elemzett Ingatlan Változó:</label>
      <select id="m_var_sel" style="width:100%; padding:9px 12px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="recalcMoran()">
        <option value="nm_ar_huf" selected>Négyzetméterár (HUF/m²) - Erős térbeli klaszter</option>
        <option value="allapot_kod">Műszaki Állapot Kód - Szignifikáns szomszédsági hatás</option>
        <option value="alapterulet_nm">Alapterület (m²) - Véletlenszerűbb méreteloszlás</option>
        <option value="szobaszam_osszes">Összes Szobaszám - Enyhe lokális összefüggés</option>
      </select>
    </div>

    <!-- Szomszédsági k csúszka -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:700; color:#334155; margin-bottom:8px; text-transform:uppercase;">
        <span>2. Szomszédok Száma (KNN k):</span> <span id="m_lbl_k" style="color:#2563eb; font-weight:800;">k = 8</span>
      </div>
      <input type="range" id="m_slider_k" min="0" max="5" value="2" step="1" style="width:100%; accent-color:#2563eb;" oninput="recalcMoran()">
      <div style="display:flex; justify-content:space-between; font-size:11px; color:#64748b; margin-top:4px;">
        <span>k=4</span><span>k=6</span><span>k=8 (Optimális)</span><span>k=10</span><span>k=12</span><span>k=16</span>
      </div>
    </div>
  </div>

  <!-- KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap:14px; margin-bottom:14px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Globális Moran\'s I</div>
      <div id="m_res_i" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">--</div>
      <div style="font-size:11px; color:#2563eb;">Térbeli autokorreláció</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Z-Statisztika</div>
      <div id="m_res_z" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">--</div>
      <div style="font-size:11px; color:#16a34a;">Szórási egység (z-score)</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">P-érték (999 perm.)</div>
      <div id="m_res_p" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">--</div>
      <div id="m_res_sig" style="font-size:11px; color:#9333ea;">Szignifikancia szint</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Térbeli Struktúra</div>
      <div id="m_res_str" style="font-size:16px; font-weight:800; color:#b45309; margin:8px 0;">--</div>
      <div style="font-size:11px; color:#d97706;">Ökonometriai következmény</div>
    </div>
  </div>

  <div id="m_res_desc" style="background:#f8fafc; border-left:4px solid #2563eb; padding:12px 16px; border-radius:6px; font-size:13px; color:#334155; line-height:1.5;">
    --
  </div>
</div>

<script>
(function() {{
  const M_DATA = {json.dumps(moran_data_js)};
  const k_vals = [4, 6, 8, 10, 12, 16];

  function update() {{
    const varKey = document.getElementById('m_var_sel').value;
    const kIdx = parseInt(document.getElementById('m_slider_k').value);
    const kVal = k_vals[kIdx] || 8;
    
    document.getElementById('m_lbl_k').innerText = 'k = ' + kVal;
    
    const info = (M_DATA[varKey] && M_DATA[varKey][kVal.toString()]) || {{ "I": 0.109, "z": 2.88, "p": 0.009 }};
    
    document.getElementById('m_res_i').innerText = info.I.toFixed(3);
    document.getElementById('m_res_z').innerText = info.z.toFixed(2);
    document.getElementById('m_res_p').innerText = info.p < 0.001 ? '< 0.001' : info.p.toFixed(3);
    
    let sigText = 'Nem szignifikáns (p >= 0.05)';
    let strText = 'Véletlenszerű eloszlás';
    let descText = '';
    
    if (info.p < 0.05) {{
      sigText = info.p < 0.01 ? 'Erősen szignifikáns (p < 0.01)' : 'Szignifikáns (p < 0.05)';
      strText = 'Pozitív Térbeli Klaszterezettség';
    }}
    
    if (varKey === 'nm_ar_huf') {{
      descText = '<b>Négyzetméterár:</b> Az ár statisztikailag szignifikáns pozitív autokorrelációt mutat (Moran I = ' + info.I.toFixed(3) + ', p = ' + info.p + '). A hasonló árfekvésű lakások földrajzilag tömörülnek (pl. Óhegy magasabb árszintje, Újhegy és Pongrácz alacsonyabb árai). Ez igazolja, hogy az egyszerű OLS becslés térbelileg torzított, és indokolja a 09-es SAR/SEM térökonometriai modellek használatát!';
    }} else if (varKey === 'allapot_kod') {{
      descText = '<b>Műszaki Állapot:</b> Szignifikáns szomszédsági hasonlóság (Moran I = ' + info.I.toFixed(3) + '). Az épületek állapota gyakran tömbönként vagy lakótelepenként homogén (pl. egy teljes paneltömb szigetelése vagy régi polgári házak közös állapota).';
    }} else if (varKey === 'alapterulet_nm') {{
      descText = '<b>Alapterület:</b> Nincs szignifikáns térbeli klaszterezettség (Moran I = ' + info.I.toFixed(3) + ', p = ' + info.p + '). Kis és nagy lakások egyaránt megtalálhatók a kerület szinte minden pontján, a lakásméret nem képez földrajzi szigeteket.';
    }} else {{
      descText = '<b>Szobaszám:</b> Alacsony, nem szignifikáns autokorreláció (Moran I = ' + info.I.toFixed(3) + '). A szobaszám eloszlása a lakásmérethez hasonlóan a teljes kerületi szövetben szétszórtan jelenik meg.';
    }}
    
    document.getElementById('m_res_sig').innerText = sigText;
    document.getElementById('m_res_str').innerText = strText;
    document.getElementById('m_res_desc').innerHTML = descText;
  }}

  window.recalcMoran = update;
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_moran_app))